In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os
import json
import time
import pickle
from captum.attr import IntegratedGradients, FeatureAblation
from captum.attr._utils.visualization import visualize_image_attr
from captum._utils.models.linear_model import SkLearnLinearRegression
import cmcrameri.cm as cmc
import matplotlib.font_manager as fm
from ResNet_BiLSTM_Transformer import SixHourlyFRDataset, ResNetBlock1D, ResNetLSTMTransformer

In [ ]:
class CaptumFeatureInteractionAnalyzer:
    """
    Analyze feature interactions using Captum's attribution methods
    """
    
    def __init__(self, model, device='cuda'):
        self.model = model
        self.device = device
        self.model.to(self.device)
        
    def _forward_func(self, inputs, target_horizon=1):
        """
        Wrapper to make model compatible with Captum
        """
        inputs = inputs.to(self.device)
        self.model.eval()
        outputs = self.model(inputs)
        return outputs[target_horizon]
    
    def compute_attributions(self, 
                           inputs: torch.Tensor,
                           baseline: torch.Tensor = None,
                           target_horizon: int = 1,
                           method: str = 'integrated_gradients',
                           n_steps: int = 50) -> torch.Tensor:
        """
        Compute feature attributions using specified method
        """
        inputs = inputs.to(self.device)
        
        if baseline is None:
            baseline = torch.zeros_like(inputs).to(self.device)
        else:
            baseline = baseline.to(self.device)
        
        with torch.enable_grad():
            was_training = self.model.training
            self.model.train()
            
            dropout_modules = []
            for module in self.model.modules():
                if isinstance(module, nn.Dropout):
                    dropout_modules.append((module, module.p))
                    module.p = 0
            
            try:
                def forward_func_wrapped(x):
                    x = x.to(self.device)
                    return self.model(x)[target_horizon]
                
                if method == 'integrated_gradients':
                    ig = IntegratedGradients(forward_func_wrapped)
                    attributions = ig.attribute(
                        inputs.requires_grad_(), 
                        baselines=baseline,
                        n_steps=n_steps,
                        internal_batch_size=32
                    )
                else:
                    raise ValueError(f"Unknown method: {method}")
                    
            finally:
                self.model.train(was_training)
                for module, p in dropout_modules:
                    module.p = p
                    
        return attributions.detach()

In [ ]:
class ValidationAnalyzer:
    """
    Analyze model interpretability on validation dataset
    """
    
    def __init__(self, 
                 model_path: str,
                 val_dataset_path: str,
                 model_config: Dict,
                 device: str = 'cuda'):
        
        self.device = device
        self.model_config = model_config
        
        # Base feature names mapped dynamically exactly like the dataset
        base_features = ['is_FR', 'tas', 'RH', 'Tw', 'WS', 'TP', 'DT']
        levels = [975, 950, 900, 850, 800, 750, 700, 650, 500]
        level_features = []
        for lvl in levels:
            level_features.extend([f'T{lvl}', f'RH{lvl}', f'VV{lvl}'])
            
        self.base_feature_names = base_features + level_features
        self.num_base_features = len(self.base_feature_names)
        
        # Create full feature names including 6h and 12h source features (102 features total)
        self.feature_names = self.base_feature_names.copy()
        
        num_model_features = self.model_config.get('num_features', 102)
        if num_model_features > self.num_base_features:
            for lag in [1, 2]: # lag1 = 6h, lag2 = 12h
                source_feature_names = [f"Source_{name}_lag{lag*6}h" for name in self.base_feature_names]
                self.feature_names.extend(source_feature_names)
        
        print(f"Model config expects {num_model_features} features")
        print(f"Feature names list generated {len(self.feature_names)} names")
        
        # Load model AFTER setting up feature names
        self.model = self._load_model(model_path)
        
        # Initialize Captum analyzer
        self.captum_analyzer = CaptumFeatureInteractionAnalyzer(self.model, device)
        
    def _load_model(self, model_path):
        """Load the trained Transformer model"""
        use_resnet = self.model_config.get('use_resnet', True)
    
        # Filter out 'use_resnet' since it's not an argument for the model's __init__
        model_params = {k: v for k, v in self.model_config.items() if k != 'use_resnet'}
        
        # Ensure num_features aligns with our dataset logic
        model_params['num_features'] = len(self.feature_names) 
    
        if use_resnet:
            # FIX: We removed lookback_steps=12 and forecast_steps=[1,2,3,4] from here
            # because they are already inside **model_params!
            model = ResNetLSTMTransformer(**model_params)
        else:
            # Fallback if needed
            model = SimpleCNNLSTM(**model_params)
    
        model.load_state_dict(torch.load(model_path, map_location=self.device, weights_only=False))
        model.to(self.device)
        model.eval()
        
        # Fix inplace operations for Captum
        for module in model.modules():
            if isinstance(module, torch.nn.ReLU):
                module.inplace = False
    
        return model
    
    def load_validation_dataset(self, data_path: str, val_years: List[int]):
        """Load validation dataset"""
        print("Loading validation dataset...")
        
        self.val_dataset = SixHourlyFRDataset(
            data_path=data_path,
            train_years=val_years,
            cache_dir='./cache_6hourly/val_analysis',
            use_source_sink=True,
            source_lag_steps=[1, 2],  # MUST MATCH: 6h and 12h lag
            source_bbox={
                "min_lat": 29.9, "max_lat": 32.4,
                "min_lon": 109.8, "max_lon":112.2
            },
            sink_bbox= 
            {"min_lat": 27.4, "max_lat": 29.8, "min_lon": 105.5, "max_lon": 120}, 
        )
        
        self.val_loader = DataLoader(
            self.val_dataset,
            batch_size=32,
            shuffle=False,
            num_workers=0
        )
        
        print(f"Loaded {len(self.val_dataset)} validation samples")
        
    def find_interesting_samples(self, n_samples: int = 10) -> Dict[str, List[int]]:
        """Find interesting samples from validation set"""
        print("Finding interesting samples...")
        
        all_predictions = {h: [] for h in [1, 2, 3, 4]}
        all_targets = {h: [] for h in [1, 2, 3, 4]}
        all_indices = []
        
        with torch.no_grad():
            for idx, batch in enumerate(self.val_loader):
                sequences = batch['sequence'].to(self.device)
                predictions = self.model(sequences)
                
                batch_size = sequences.shape[0]
                start_idx = idx * self.val_loader.batch_size
                
                for step in [1, 2, 3, 4]:
                    probs = torch.sigmoid(predictions[step]).cpu().numpy()
                    targets = batch['targets'][step].cpu().numpy()
                    
                    all_predictions[step].extend(probs.squeeze().tolist())
                    all_targets[step].extend(targets.tolist())
                
                all_indices.extend(range(start_idx, start_idx + batch_size))
        
        interesting_samples = {
            'all_samples': all_indices[:min(n_samples * 5, len(all_indices))]
        }
        
        print(f"\nTotal samples found: {len(interesting_samples['all_samples'])}")
        return interesting_samples


def analyze_specific_source_sink_interactions(analyzer: ValidationAnalyzer,
                                            sample_indices: List[int],
                                            output_dir: str,
                                            n_samples: int = 100):
    """
    Analyze interactions between specific source and sink variables for all forecast horizons
    """
    feature_names = analyzer.feature_names
    
    # DYNAMIC INDEXING: Find exact indices so it never crashes!
    sink_var_names = ['tas', 'Tw', 'WS', 'DT', 'T975', 'T950', 'T900', 'T850', 
                      'T800', 'T750', 'T700', 'T650', 'T500', 'RH950', 'RH650']
    
    source_var_names = ['tas','Tw', 'WS', 'TP', 'DT', 'T500', 'T650', 'T700', 'T750', 
                        'T800', 'T850', 'T900', 'T950', 'T975']
    
    # Build dictionary mappings securely
    sink_vars = {name: feature_names.index(name) for name in sink_var_names if name in feature_names}
    
    source_vars = {}
    for name in source_var_names:
        # We specifically want the 12h lag (lag2)
        target_name = f"Source_{name}_lag12h"
        if target_name in feature_names:
            # We map 'WbT_12h' to the actual integer index in the 102-feature list
            source_vars[f"{name}_12h"] = feature_names.index(target_name)
    
    # Limit samples
    n_samples = min(n_samples, len(sample_indices))
    selected_indices = sample_indices[:n_samples]
    
    print(f"\nAnalyzing {n_samples} samples for all forecast horizons...")
    
    # Storage for all horizons
    all_horizon_attributions = {
        '6h': [], '12h': [], '18h': [], '24h': []
    }
    
    batch_size = 10
    
    # Process samples
    for batch_start in tqdm(range(0, n_samples, batch_size), desc="Processing samples"):
        batch_end = min(batch_start + batch_size, n_samples)
        batch_indices = selected_indices[batch_start:batch_end]
        
        batch_samples = []
        for idx in batch_indices:
            sample_data = analyzer.val_dataset[idx]
            batch_samples.append(sample_data['sequence'])
        
        if len(batch_samples) == 0:
            continue
            
        batch_tensor = torch.stack(batch_samples).to(analyzer.device)
        baseline = torch.zeros_like(batch_tensor).to(analyzer.device)
        
        # Get attributions for all horizons
        for horizon_idx, horizon_name in enumerate(['6h', '12h', '18h', '24h'], 1):
            try:
                attrs = analyzer.captum_analyzer.compute_attributions(
                    batch_tensor, baseline, target_horizon=horizon_idx, n_steps=30
                )
                attrs_cpu = attrs.detach().cpu().numpy()
                all_horizon_attributions[horizon_name].extend(attrs_cpu)
            except Exception as e:
                print(f"Error for {horizon_name}: {e}")
    
    # First, calculate interaction scores to determine top pairs
    all_interaction_scores = calculate_all_interaction_scores(
        all_horizon_attributions, sink_vars, source_vars
    )
        
    # Create temporal heatmap showing patterns over time
    create_temporal_attribution_heatmap(
        all_horizon_attributions, all_interaction_scores,
        sink_vars, source_vars, output_dir, n_samples
    )
    
    # Create interaction lollipop plots
#    create_source_sink_interaction_analysis(
#        all_horizon_attributions, sink_vars, source_vars, output_dir
#    )
    
    # Create horizon importance comparison
    analyze_forecast_horizon_importance_detailed(
        all_horizon_attributions, analyzer.feature_names, output_dir
    )


def calculate_all_interaction_scores(all_attributions, sink_vars, source_vars):
    """
    Calculate interaction scores for all variable pairs across all horizons
    """
    all_scores = {}
    
    for horizon_name in ['6h', '12h', '18h', '24h']:
        if horizon_name not in all_attributions or len(all_attributions[horizon_name]) == 0:
            continue
        
        attrs = np.array(all_attributions[horizon_name])
        
        # FIX: Ensure shape is (batch_size, seq_len, num_features)
        if attrs.shape[1] > attrs.shape[2]:
            attrs = np.transpose(attrs, (0, 2, 1))
            
        interaction_scores = {}
        
        for sink_name, sink_idx in sink_vars.items():
            for source_name, source_idx in source_vars.items():
                
                # Check array bounds safely
                if source_idx >= attrs.shape[2] or sink_idx >= attrs.shape[2]:
                    continue
                
                # Calculate interaction metric
                sink_attr = np.abs(attrs[:, :, sink_idx]).flatten()
                source_attr = np.abs(attrs[:, :, source_idx]).flatten()
                
                if np.std(sink_attr) > 0 and np.std(source_attr) > 0:
                    correlation = np.corrcoef(sink_attr, source_attr)[0, 1]
                    importance = (np.mean(sink_attr) + np.mean(source_attr)) / 2
                    score = abs(correlation) * importance
                else:
                    score = 0
                
                interaction_scores[(sink_name, source_name)] = score
        
        all_scores[horizon_name] = interaction_scores
    
    return all_scores


def create_temporal_attribution_heatmap(all_attributions, all_interaction_scores,
                                       sink_vars, source_vars,
                                       output_dir, n_samples):
    """
    Create heatmaps showing temporal patterns for top variable pairs for each forecast horizon
    """
    fig, axes = plt.subplots(2, 2, figsize=(20, 16))
    axes = axes.ravel()
    
    horizons = ['6h', '12h', '18h', '24h']
    
    try:
        sns.set_style("white")
    except:
        pass
    
    for idx, (horizon_name, ax) in enumerate(zip(horizons, axes)):
        if horizon_name not in all_interaction_scores or horizon_name not in all_attributions:
            ax.text(0.5, 0.5, f'No data for {horizon_name}', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_xticks([])
            ax.set_yticks([])
            continue
        
        horizon_scores = all_interaction_scores[horizon_name]
        top_15_pairs = sorted(horizon_scores.items(), key=lambda x: x[1], reverse=True)[:15]
        
        horizon_attrs = np.array(all_attributions[horizon_name])
        
        # FIX: Ensure shape is (batch_size, seq_len, num_features)
        if horizon_attrs.shape[1] > horizon_attrs.shape[2]:
            horizon_attrs = np.transpose(horizon_attrs, (0, 2, 1))
        
        heatmap_data = []
        pair_labels = []
        
        for (sink_name, source_name), score in top_15_pairs:
            sink_idx = sink_vars.get(sink_name)
            source_idx = source_vars.get(source_name)
            
            if sink_idx is None or source_idx is None:
                continue
                
            if sink_idx >= horizon_attrs.shape[2] or source_idx >= horizon_attrs.shape[2]:
                continue
            
            sink_attr_time = np.abs(horizon_attrs[:, :, sink_idx]).mean(axis=0)
            source_attr_time = np.abs(horizon_attrs[:, :, source_idx]).mean(axis=0)
            combined_attr = (sink_attr_time + source_attr_time) / 2
            
            heatmap_data.append(combined_attr)
            pair_labels.append(f"{sink_name}-{source_name}")
        
        heatmap_data = np.array(heatmap_data)
        
        # Failsafe for empty data
        if len(heatmap_data.shape) < 2 or heatmap_data.shape[0] == 0:
            ax.text(0.5, 0.5, f'No interaction pairs found for {horizon_name}', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_xticks([])
            ax.set_yticks([])
            continue
            
        n_timesteps = heatmap_data.shape[1]
        time_labels = [f"t-{(n_timesteps-i)*6}h" for i in range(n_timesteps)]
        
        df_heatmap = pd.DataFrame(heatmap_data, 
                                 index=pair_labels,
                                 columns=time_labels)
        
        font_kwargs = {'fontproperties': chinese_font} if 'chinese_font' in globals() else {}
        cmap_choice = cmc.navia if 'cmc' in globals() else 'viridis'
        
        sns.heatmap(df_heatmap, 
                   ax=ax,
                   cmap=cmap_choice,
                   cbar_kws={'label': 'Average Attribution', 
                            'fraction': 0.046, 
                            'pad': 0.04},
                   linewidths=0.5,
                   linecolor='gray',
                   square=False,
                   xticklabels=True,
                   yticklabels=True,
                   vmin=0,
                   cbar=True,
                   alpha=0.75)

        try:
            cbar = ax.collections[0].colorbar
            cbar.set_label('|Average Attribution|', **font_kwargs)
            cbar.ax.tick_params(labelsize=11)
            cbar.ax.yaxis.label.set_fontsize(11)
        except:
            pass

        ax.set_title(f'{horizon_name} Forecasting', fontsize=18, pad=10, **font_kwargs)
        ax.set_xlabel('Time before Forecasting', fontsize=16, **font_kwargs)
        ax.set_ylabel('Variable Pairs', fontsize=16, **font_kwargs)
        
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=15)
        ax.set_yticklabels(ax.get_yticklabels(), fontsize=15)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    save_path = os.path.join(output_dir, 'temporal_attribution_heatmap.png')
    plt.savefig(save_path, dpi=600, bbox_inches='tight')
    plt.close()
    
    try:
        sns.reset_defaults()
    except:
        pass
    print(f"Saved temporal attribution heatmap to {save_path}")


def analyze_forecast_horizon_importance_detailed(all_attributions, feature_names, output_dir):
    """
    Create detailed dumbbell charts for top 15 features at each forecast horizon.
    Features a widening shade connecting the t-72h point (small) to the max attribution point (large).
    """
    FORECAST_COLORS = {'6h': '#1f77b4', '12h': '#ff7f0e', '18h': '#2ca02c', '24h': '#d62728'}
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    axes = axes.ravel()
    
    horizons = ['6h', '12h', '18h', '24h']
    
    for idx, (horizon_name, ax) in enumerate(zip(horizons, axes)):
        if horizon_name not in all_attributions or len(all_attributions[horizon_name]) == 0:
            continue
        
        attrs = np.array(all_attributions[horizon_name])
        
        # Ensure shape is (batch_size, seq_len, num_features)
        if attrs.shape[1] > attrs.shape[2]:
            attrs = np.transpose(attrs, (0, 2, 1))
        
        mean_attrs_over_batch = np.abs(attrs).mean(axis=0)
        attr_72h = mean_attrs_over_batch[0, :]
        attr_max = mean_attrs_over_batch.max(axis=0)
        avg_importance = mean_attrs_over_batch.mean(axis=0)
        
        feature_data = []
        for i, importance in enumerate(avg_importance):
            name = feature_names[i]
            
            if 'is_FR' in name:
                continue
                
            # ==========================================
            # THE FIX: Strip out both "Source_" and "lag"
            # Turns "Source_Tw_lag12h" into "Tw_12h"
            # ==========================================
            if name.startswith("Source_"):
                name = name.replace("Source_", "").replace("lag", "")
            
            feature_data.append({
                'name': name,
                'val_72h': attr_72h[i],
                'val_max': attr_max[i],
                'avg_imp': importance
            })
        
        sorted_features = sorted(feature_data, key=lambda x: x['avg_imp'], reverse=True)[:15]
        
        if sorted_features:
            features = [f['name'] for f in sorted_features]
            vals_72h = [f['val_72h'] for f in sorted_features]
            vals_max = [f['val_max'] for f in sorted_features]
            
            y_pos = np.arange(len(features))
            horizon_color = FORECAST_COLORS.get(horizon_name, '#1f77b4')
            
            r_small = 0.04
            r_large = 0.22
            
            for i, (v_72h, v_max) in enumerate(zip(vals_72h, vals_max)):
                X = [v_72h, v_72h, v_max, v_max]
                Y = [i - r_small, i + r_small, i + r_large, i - r_large]
                ax.fill(X, Y, color=horizon_color, alpha=0.5, zorder=1)
                
                ax.scatter(v_72h, i, s=50, color='#2F4F4F', edgecolors='white', 
                          linewidth=0.8, zorder=3)
                ax.scatter(v_max, i, s=200, c=[horizon_color], edgecolors='white', 
                          linewidth=1, alpha=0.95, zorder=4)
            
            ax.set_yticks(y_pos)
            ax.set_yticklabels(features, fontsize=14)
            
            font_kwargs = {'fontproperties': chinese_font} if 'chinese_font' in globals() else {}
            
            ax.set_xlabel('|Attribution Value|', fontsize=14, **font_kwargs)
            ax.set_title(f'{horizon_name} Forecasting - Top 15 Feature Variations', fontsize=14, **font_kwargs)
            
            ax.grid(axis='x', color='lightgray', linestyle='--', linewidth=0.5, zorder=0)
            ax.tick_params(labelsize=14)
            ax.set_xlim(left=0)
            
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            
            for i, v_max in enumerate(vals_max):
                ax.text(v_max + ax.get_xlim()[1]*0.02, i, f'{v_max:.3f}', 
                       va='center', fontsize=9, alpha=0.8)
    
    plt.tight_layout()
    save_path = os.path.join(output_dir, 'forecast_horizon_importance_dumbbell.png')
    plt.savefig(save_path, dpi=600, bbox_inches='tight')
    plt.close()
    print(f"Saved detailed dumbbell chart to {save_path}")

In [ ]:
def run_source_sink_analysis():
    """
    Run only the source-sink interaction analysis with heatmaps and lollipop charts
    """
    # Configuration
    model_config = {
        'num_features': 102,  # 34 base + 34 (6h) + 34 (12h)
        'lookback_steps': 12,
        'forecast_steps': [1, 2, 3, 4],
        'hidden_dim': 128,
        'lstm_layers': 3,
        'resnet_layers': [3, 4, 5],
        'base_channels': 32,
        'dropout': 0.2,
        'num_heads': 4,
        'use_resnet': True,
        'use_skip_connection': True  
    }
    
    # Paths
    model_path = r'E:\PhD\freezing rains\Early-warning\model\O4(hybrid)\best_source_model.pth'
    data_path = r"E:\PhD\freezing rains\Early-warning\2024\final\2024_FEB_final_predictions.csv"
    output_dir = r'E:\PhD\freezing rains\Early-warning\fig\interactions\2024'
    
    os.makedirs(output_dir, exist_ok=True)
    
    print("Initializing analyzer...")
    analyzer = ValidationAnalyzer(
        model_path=model_path,
        val_dataset_path=data_path,
        model_config=model_config
    )
    
    print("Loading validation dataset...")
    analyzer.load_validation_dataset(data_path, val_years=[2024])
    
    print("Finding samples...")
    interesting_samples = analyzer.find_interesting_samples(n_samples=300)
    all_indices = interesting_samples['all_samples']
    
    print(f"Total samples available for analysis: {len(all_indices)}")
    
    if len(all_indices) > 0:
        print("\n" + "="*60)
        print("Running source-sink interaction analysis...")
        print("="*60)
        
        analyze_specific_source_sink_interactions(
            analyzer,
            all_indices,
            output_dir,
            n_samples=300  
        )
    
    print(f"\n" + "="*60)
    print(f"Analysis complete! Check '{output_dir}' for results:")
    print("- temporal_attribution_heatmap.png: Attribution patterns for all forecast horizons")
    print("- source_sink_interactions.png: Top interactions for each horizon")
    print("- forecast_horizon_importance.png: Top 15 features for each horizon")
    print("="*60)
    
    return analyzer


if __name__ == "__main__":
    analyzer = run_source_sink_analysis()